In [ ]:
import geopandas as gpd
bldg=
pop=

bldg = bldg.to_crs(pop.crs)


In [ ]:
bldg_sal = gpd.sjoin(
    bldg,
    pop[["EA_CODE", "sal2023_est", "geometry"]],
    how="inner",
    predicate="intersects"
)



sal_area = bldg_sal.groupby("EA_CODE")["area_in_meters"].sum().reset_index()
sal_area = sal_area.rename(columns={"bldg_area": "sal_bldg_area"})

In [ ]:
bldg_sal = bldg_sal.merge(sal_area, on="EA_CODE", how="left")

In [ ]:
bldg_sal["pop_alloc"] = (
    bldg_sal["sal2023_est"] *
    (bldg_sal["bldg_area"] / bldg_sal["sal_bldg_area"])
)

bldg_sal[["EA_CODE", "bldg_area", "pop_alloc"]].head()

In [ ]:
##this is to check the difference...difference should be very small

check = bldg_sal.groupby("EA_CODE")["pop_alloc"].sum().reset_index()

check = check.merge(pop[["EA_CODE", "sal2023_est"]], on="EA_CODE")

print(check.head())

In [ ]:
def areametric(bldg, pop):
    b = bldg.copy()
    b["area"] = b.geometry.area

    b = gpd.sjoin(
        b,
        pop[["EA_CODE", "sal2023_est", "geometry"]],
        predicate="intersects"
    )

    sal_sum = b.groupby("EA_CODE")["area"].sum().rename("sal_area")
    b = b.merge(sal_sum, on="EA_CODE")

    b["est_pop"] = b["sal2023_est"] * (b["area"] / b["sal_area"])
    return b

In [ ]:
filters = [0, 5, 10, 15, 20, 25, 30, 35, 40]

outputs = {}
for f in filters:
    b_f = bldg[bldg.geometry.area >= f]
    outputs[f] = areametric(b_f, pop)

In [ ]:
sal_results = {}

for f, df in outputs.items():
    sal_results[f] = df.groupby("SAL_ID")["est_pop"].sum()

In [ ]:
import numpy as np

def rmse(true, pred):
    return np.sqrt(np.mean((true - pred) ** 2))

In [ ]:
rmse_results = {}

for f, est in sal_results.items():
    merged = pop.set_index("SAL_ID")["SAL2023"].align(est, join="inner")[0]
    rmse_results[f] = rmse(merged, est)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
r2_results = {}

for f, est in sal_results.items():

    merged = pop.set_index("SAL_ID")["SAL2023"].to_frame().join(est.rename("est_pop")).dropna()

    X = merged["SAL2023"].values.reshape(-1,1)
    y = merged["est_pop"].values

    model = LinearRegression().fit(X, y)
    pred = model.predict(X)

    r2_results[f] = r2_score(y, pred)

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "filter_m2": list(filters),
    "RMSE": [rmse_results[f] for f in filters],
    "R2": [r2_results[f] for f in filters]
})

summary